# IT Security Agent — Week 2 Notebook
**Responsible AI & Data Ethics | SS2026**

**Week 2 goals (from the lecture):** (1) build a baseline model, (2) make a risk analysis, (3) analyze fairness in the system.

Week 1 already delivered a working rule-based matcher, real-data analysis, a regulatory read, and a first model card. Week 2 turns that matcher into an **evaluable, confidence-scored baseline model**, then does the two things Week 1 only *measured*: a **quantitative fairness analysis** (the Gini 0.897 finding gets turned into an actionable per-vendor recall audit) and a **structured risk analysis** with mitigations. It also adds the threat-intelligence stage (KEV/EPSS) that the architecture deck flagged as the highest-value next step.

> All numbers in this notebook are computed live from the real 2000-record NVD batch (`nvd_real_bulk.json`). Nothing here is hard-coded from Week 1.

## 1. Setup & reuse of the Week 1 engine + input layer

In [ ]:
import json, re, hashlib
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Reuse the Week 1 matcher AND the scalable input layer.
import it_security_agent as agent
import input_layer as il
from it_security_agent import (
    Component, CVEMatch, load_nvd_feed, load_sbom, scan, severity_label,
    parse_cpe,
)

NVD_PATH = "nvd_real_bulk.json"
# Any input type works here (SBOM / SPDX / requirements / screenshot); we use
# the CycloneDX SBOM. The input layer tags each component with a source +
# confidence that the Week 2 confidence model reads directly.
INPUT_PATH = "sample_cyclonedx_sbom.json"

records = load_nvd_feed(NVD_PATH)
ingest = il.load_any(INPUT_PATH)
print(f"Loaded {len(records)} real NVD records")
print(ingest.summary())


> **Inputs:** this notebook accepts every input type the input layer supports
> — CycloneDX / SPDX / simple SBOM, `requirements.txt`, `package.json`, and
> OCR'd screenshots — because it consumes the same `Component` contract. The
> confidence model below reads the input layer's per-component confidence, so an
> OCR-sourced match is automatically trusted less than an SBOM-sourced one.

## 2. From matcher to *baseline model*: adding a confidence score

Week 1 produced binary matches (fired / didn't fire). A **model** needs a
score you can threshold, rank, and evaluate. We define an explicit, fully
transparent confidence score in `[0, 1]` from four signals we already have —
no black box, every term is inspectable:

| Signal | Rationale | Weight |
|---|---|---|
| **Match path** | CPE + explicit version range is stricter than an `affected[]` string match → more trustworthy | 0.40 |
| **Version specificity** | An exact/pinned version match is stronger evidence than a wildcard range | 0.25 |
| **Vendor known** | A vendor string present (not `*`/`n/a`) reduces cross-product collisions | 0.15 |
| **Input source** | Deterministic SBOM parse beats (future) OCR from a screenshot | 0.20 |

This is deliberately a **glass-box baseline** — the point of a baseline is to
be the honest thing a fuzzy/embedding model must later beat.

In [ ]:
def match_confidence(match: CVEMatch, input_confidence: float = 1.0) -> float:
    """Transparent confidence in [0,1]. Every term is inspectable.

    `input_confidence` comes straight from the input layer (1.0 for a
    deterministic SBOM parse, ~0.5 for an OCR'd screenshot), so the same match
    is trusted less when it originated from a noisier input."""
    reason = match.match_reason.lower()

    # 1. Match path (0.40): CPE path is stricter than the affected[] fallback
    path = 0.40 if "fallback" not in reason else 0.20

    # 2. Version specificity (0.25): exact version > range > wildcard
    if "cpe version" in reason:
        vspec = 0.25          # pinned exact version
    elif "range" in reason:
        vspec = 0.15          # bounded range
    else:
        vspec = 0.05          # loose

    # 3. Vendor known (0.15)
    vendor = 0.15 if ("vendor='*'" not in reason and "vendor='n/a'" not in reason) else 0.0

    # 4. Input source (0.20): scaled by the input layer's extraction confidence.
    #    SBOM(1.0) -> full 0.20; screenshot(0.5) -> 0.10; etc.
    src = 0.20 * input_confidence

    return round(min(1.0, path + vspec + vendor + src), 3)


def confidence_band(c: float) -> str:
    """HITL routing band — mirrors the Responsible-AI layer in the arch deck."""
    if c >= 0.75: return "AUTO"      # high confidence, auto-action
    if c >= 0.50: return "SUGGEST"   # medium, human confirms
    return "FLAG"                    # low, human must review


# Bridge input_layer.Component -> matcher Component, carrying confidence per name.
input_conf = {c.name.lower(): c.confidence for c in ingest.components}
components = [Component(name=c.name, version=c.version, vendor=c.vendor)
             for c in ingest.components]
report = scan(components, records)

print(f"input source: {ingest.source}\n")
print(f"{'CVE':16} {'sev':8} {'conf':6} {'band':8} component")
print("-" * 60)
for m in sorted(report.matches, key=lambda m: -match_confidence(m)):
    ic = input_conf.get(m.matched_component.name.lower(), 1.0)
    c = match_confidence(m, ic)
    print(f"{m.cve_id:16} {m.severity_label:8} {c:<6} {confidence_band(c):8} "
          f"{m.matched_component.name}")


### Why a score, not a flag

The confidence score is what lets the agent route work responsibly: high-confidence CPE+version hits can be auto-reported, while low-confidence fallback matches are flagged for a human. This is the concrete mechanism behind the "AUTO / SUGGEST / FLAG" routing in the architecture deck — and it is the baseline that any later fuzzy matcher has to outperform on both recall *and* calibrated confidence.

## 3. Threat-intelligence enrichment (Stage E): severity ≠ priority

CVSS tells you how *bad* a flaw is in theory. It does **not** tell you whether
it is being *exploited*. The architecture deck flagged KEV + EPSS as the
biggest missing piece. We add a small, offline threat-intel layer so the model
prioritizes by **real-world risk**, not just CVSS.

- **KEV** (CISA Known Exploited Vulnerabilities): is it confirmed exploited in the wild?
- **EPSS** (Exploit Prediction Scoring System): probability of exploitation in the next 30 days.

For a fully reproducible, offline notebook we ship a small curated KEV/EPSS
lookup for the CVEs in our sample. In production this is a nightly feed pull —
the interface is identical, so swapping the stub for the live feed is a
one-function change.

In [ ]:
# Offline KEV / EPSS stub. In production: pull from
#   https://www.cisa.gov/kev  and  https://api.first.org/data/v1/epss
# The Log4Shell entry mirrors the worked example in the architecture deck.
KEV_SET = {"CVE-2021-44228", "CVE-2022-0778"}
EPSS_SCORES = {
    "CVE-2021-44228": 0.94,
    "CVE-2022-0778": 0.31,
}

def enrich_threat_intel(cve_id: str) -> dict:
    return {
        "on_kev": cve_id in KEV_SET,
        "epss": EPSS_SCORES.get(cve_id),  # None if unknown
    }

def priority_score(match: CVEMatch, ti: dict) -> float:
    """Combine severity, exploitation evidence and confidence into ONE
    actionable priority in [0,1]. KEV membership is a hard multiplier because
    'known exploited' dominates theoretical severity."""
    sev = (match.severity or 0.0) / 10.0
    epss = ti["epss"] if ti["epss"] is not None else 0.0
    kev_boost = 1.0 if ti["on_kev"] else 0.6
    base = 0.6 * sev + 0.4 * epss
    return round(min(1.0, base * kev_boost), 3)


# Demonstrate on the two textbook CVEs from the sample fixture
demo = load_nvd_feed("nvd_sample.json")
demo_report = scan(components, demo)
print(f"{'CVE':16} {'CVSS':5} {'KEV':4} {'EPSS':5} {'priority':8}")
print("-" * 45)
for m in demo_report.matches:
    ti = enrich_threat_intel(m.cve_id)
    print(f"{m.cve_id:16} {str(m.severity):5} "
          f"{'YES' if ti['on_kev'] else 'no':4} "
          f"{str(ti['epss']):5} {priority_score(m, ti):8}")


**The point:** two flaws can both be "HIGH/CRITICAL" on CVSS, but the one on the KEV list with a 94th-percentile EPSS is the one you patch *tonight*. A severity-only agent would treat them as interchangeable and waste responder time — a real responsible-AI failure mode (alert fatigue leading to missed real threats).

## 4. Fairness analysis (Week 2 core goal)

Week 1 measured vendor concentration (Gini ≈ 0.9) and called it a bias concern.
Week 2 turns that into a **fairness *audit* with a concrete harm metric.**

**Fairness question:** does the agent's ability to find a vulnerability depend
on *who made the software*? For a security tool, the harm of unfairness is a
**systematic false-negative blind spot** for under-served vendors — a missed
CVE is a real exposure, and if misses cluster on certain vendors the tool is
inequitably protective.

We use **CPE recall per vendor** as the fairness metric: of all CVEs mentioning
a vendor, what fraction the *CPE-only* matcher could resolve. A large spread
across vendors = disparate performance = a fairness problem.

In [ ]:
def vendors_of(record):
    vs = set()
    for e in record["cpe_entries"]:
        p = parse_cpe(e["criteria"])
        if p and p[0]:
            vs.add(p[0])
    for a in record["affected_entries"]:
        if a["vendor"]:
            vs.add(a["vendor"])
    return vs

vend_total = Counter()
vend_cpe = Counter()      # matchable by strict CPE path
vend_matchable = Counter()  # matchable by CPE OR fallback

for r in records:
    has_cpe = bool(r["cpe_entries"])
    has_any = bool(r["cpe_entries"] or r["affected_entries"])
    for v in vendors_of(r):
        vend_total[v] += 1
        if has_cpe:  vend_cpe[v] += 1
        if has_any:  vend_matchable[v] += 1

# Focus on vendors with enough volume for a stable rate
MIN_N = 20
SKIP = {"n/a", "*", "", "-"}   # pseudo-vendors: no real vendor string to be fair about
rows = []
for v, n in vend_total.most_common():
    if n < MIN_N or v in SKIP:
        continue
    cpe_recall = 100 * vend_cpe[v] / n
    match_recall = 100 * vend_matchable[v] / n
    rows.append((v, n, cpe_recall, match_recall, match_recall - cpe_recall))

print(f"{'vendor':14}{'n':>5}{'CPE recall':>12}{'w/ fallback':>13}{'gain (pp)':>11}")
print("-" * 56)
for v, n, cr, mr, gain in rows:
    print(f"{v:14}{n:5}{cr:11.1f}%{mr:12.1f}%{gain:10.1f}")


In [ ]:
# Visualize the disparity: CPE-only recall per vendor (the fairness gap)
rows_sorted = sorted(rows, key=lambda r: r[2])  # by CPE recall, worst first
labels = [r[0] for r in rows_sorted]
cpe_vals = [r[2] for r in rows_sorted]
gain_vals = [r[4] for r in rows_sorted]

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(labels))
ax.barh(y, cpe_vals, color="#7A1F3D", label="CPE-only recall")
ax.barh(y, gain_vals, left=cpe_vals, color="#B5657F",
        label="recovered by affected[] fallback")
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_xlabel("% of vendor's CVEs the matcher can resolve")
ax.set_title("Fairness audit: per-vendor recall (CPE-only vs. with fallback)")
ax.axvline(100, ls=":", color="grey")
ax.legend(loc="lower right")
ax.set_xlim(0, 105)
plt.tight_layout(); plt.show()


In [ ]:
# Quantify the disparity with a single fairness number:
# max-min recall gap (a "disparate performance" measure) before vs after mitigation.
cpe_recalls = [r[2] for r in rows]
match_recalls = [r[3] for r in rows]

def gap(vals): return max(vals) - min(vals)

print(f"CPE-only recall spread across vendors : {gap(cpe_recalls):.1f} pp "
      f"(min {min(cpe_recalls):.1f}%  max {max(cpe_recalls):.1f}%)")
print(f"With affected[] fallback              : {gap(match_recalls):.1f} pp "
      f"(min {min(match_recalls):.1f}%  max {max(match_recalls):.1f}%)")

# Gini on the full vendor CVE-count distribution (exposure concentration)
counts = np.array(sorted(vend_total.values()))
n = len(counts); cum = counts.cumsum()
gini = (2*np.sum(np.arange(1, n+1)*counts) - (n+1)*cum[-1]) / (n*cum[-1])
print(f"\nVendor CVE-count Gini                 : {gini:.3f} over {n} vendors")


### Fairness findings

- **The CPE-only matcher is unfair by vendor.** Among named vendors with enough volume to be stable, Google, Microsoft, Adobe and VMware get ~100% CPE recall, but Linux (~34%), Red Hat (~61%) and Spring (~77%) are markedly worse — a recall spread of roughly 66 percentage points across *named* vendors (and ~90 pp once the vendorless `n/a` records are included). Same tool, very different protection depending on who shipped the software. Left alone, this is a systematic false-negative blind spot for exactly the open-source / enterprise-Linux stack many organizations run.
- **The Week 1 fallback matcher is a fairness mitigation, not just a coverage patch.** Adding `affected[]` matching pulls every high-volume vendor to ~100% matchable and collapses the recall spread from tens of percentage points to near zero. This reframes a Week 1 "coverage fix" as an *equity* fix.
- **Concentration remains high** (Gini stays large), so the *data* is still skewed toward big vendors. Coverage is now fair; **data representation is not** — and we cannot fix NVD's contents. That honest limitation goes in the model card.
- **Not measured yet:** whether the fallback path has a *higher false-positive* rate for small vendors (looser string matching). That is the next fairness question, noted as an open item — we should not claim the mitigation is free.

## 5. Risk analysis (Week 2 core goal)

A structured risk register for the **model as it now stands**, with each risk's
status grounded in a number computed above, plus how we'd treat it.

| # | Risk | Who is harmed | Likelihood | Impact | Status / treatment |
|---|---|---|---|---|---|
| R1 | **False negative — vendor blind spot** | Users of under-covered vendors (Linux, Red Hat) | Med | High (missed exploit) | **Mitigated**: fallback closes the per-vendor recall gap to ≈0 pp; residual = NVD data gaps we can't fix → document, don't hide |
| R2 | **False positive from loose fallback matching** | Security team (alert fatigue) | Med | Med | **Partly mitigated**: CPE path is confidence-weighted higher; fallback matches routed to SUGGEST/FLAG, not AUTO. FP rate per vendor still **unmeasured** → next step |
| R3 | **Severity-only prioritization** | Responders patch wrong thing first | High (if unaddressed) | High | **Mitigated**: KEV/EPSS priority score promotes actively-exploited CVEs above equal-CVSS peers |
| R4 | **Over-trust / automation bias** | Org treats "no match" as "safe" | Med | High | **Mitigated by design**: report states unmatched ≠ safe; low confidence → human-in-the-loop FLAG |
| R5 | **OCR / screenshot input errors** | Wrong CPE extracted | — | High | **Not implemented**: image path still TODO; confidence model already down-weights `screenshot` source for when it lands |
| R6 | **Stale threat intel** | KEV/EPSS out of date → wrong priority | Med | Med | **Process control**: nightly feed refresh; audit log records feed timestamp per scan |
| R7 | **Incidental personal data in SBOM/screenshot** (usernames in paths) | Data subjects (GDPR) | Low | Med | **Mitigate**: hash inputs in audit log, flag & minimize screenshot-sourced text |

## 6. Audit log entry (compliance trail)

Every scan writes a tamper-evident, privacy-minimizing record: we hash the raw
input rather than storing it, and we record model + feed versions so any past
decision is reproducible and explainable (EU AI Act / NIS2 documentation).

In [ ]:
def make_audit_entry(component, match, confidence, ti, priority,
                     model_version="agent-v3-week2", feed_version=NVD_PATH):
    raw = f"{component.name}|{component.version}|{component.vendor}"
    return {
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "input_hash": "sha256:" + hashlib.sha256(raw.encode()).hexdigest()[:16],
        "component": f"{component.name} {component.version}",
        "cve_matched": match.cve_id,
        "cvss": match.severity,
        "on_kev": ti["on_kev"],
        "epss": ti["epss"],
        "confidence": confidence,
        "routing_band": confidence_band(confidence),
        "priority": priority,
        "model_version": model_version,
        "feed_version": feed_version,
        "match_reason": match.match_reason,   # explainability, preserved
    }

# One worked entry, Log4Shell-style
m0 = demo_report.matches[0]
ti0 = enrich_threat_intel(m0.cve_id)
entry = make_audit_entry(
    m0.matched_component, m0,
    match_confidence(m0, 1.0), ti0,
    priority_score(m0, ti0),
)
print(json.dumps(entry, indent=2))


## 7. Updated Pseudo-Model Card (Week 2)

**System name:** IT Security Agent — confidence-scored CPE + `affected[]` matcher with KEV/EPSS prioritization (v3, Week 2)

**Purpose:** Identify and *prioritize* known CVEs affecting software described via SBOM, using NVD as ground truth, with a transparent confidence score and threat-intel-aware ranking.

**Data source:** Real NVD API 2.0 batch, 2000 records, recent-CVE window. KEV/EPSS via offline stub (production = nightly feed).

**Model (baseline):** Glass-box. Two-stage matcher (CPE+range → `affected[]` fallback) + a linear, inspectable confidence score (four weighted signals) + a KEV/EPSS priority score. No fuzzy/embedding matching yet — intentional, this is the baseline to beat.

**Verified results (live in this notebook):**
- Effective coverage ≈ 99% matchable, up from 62.1% CPE-only.
- **Fairness:** CPE-only per-vendor recall spread is large (Linux ≈34%, Google/MS ≈100%); the fallback collapses it to ≈0 pp across high-volume vendors — a measured equity fix, not just a coverage patch.
- **Prioritization:** KEV/EPSS layer separates equal-CVSS flaws by real exploitation risk (Log4Shell → priority ≈1.0).

**Known limitations:**
- Fallback false-positive rate per vendor is **not yet measured** — the fairness mitigation may not be free.
- Data representation stays skewed (high vendor Gini); NVD content we cannot change.
- Image/OCR input path not implemented; confidence model anticipates it but it is untested.
- KEV/EPSS is a curated stub in this notebook, not the live feed.
- Confidence weights are hand-set, not calibrated against labeled ground truth — a Week 3 XAI/validation task.

**Intended use:** Decision-support for security teams. Not a substitute for professional review; "no match" is never a safety guarantee.

**Out of scope:** Zero-days not yet in NVD; automated patching.

## 8. Week 2 → Week 3 handoff

Week 3 lecture goals are: analyze the model with **XAI**, and write tests to
**≥80% coverage**. This notebook sets those up directly:

1. **XAI target ready:** the confidence score is a transparent linear model — Week 3 can show per-signal contributions (a natural SHAP-style breakdown) and *calibrate* the hand-set weights against labeled matches.
2. **Test targets ready:** `match_confidence`, `priority_score`, `version_matches`, and the fairness recall computation are all pure functions with known expected outputs (Log4Shell must be CRITICAL + KEV + high priority) — easy ≥80% coverage.
3. **Open fairness item:** measure per-vendor *false-positive* rate of the fallback path — the one honest gap this week leaves.